# LITHOS -- Phase 5: Real-Time Alert Engine
### Landslide Intelligence using Temporal & Hyperlocal Observation System

---
**What Phase 5 builds:**

| Component | What it does | Tech |
|---|---|---|
| Hourly Scheduler | Fetches live weather every hour | APScheduler |
| Live Inference | Runs fusion model on all 32,092 cells | PyTorch |
| Alert Engine | Detects when cells cross RED threshold | Python |
| Alert History | Stores all alerts with timestamps | SQLite |
| WebSocket Server | Pushes live updates to web clients | FastAPI |
| Exportable Modules | Clean .py files ready for Phase 7 | Python |

---
> Before running: Drive must have LITHOS/Phase4_data | Enable T4 GPU | Run All


## Step 1 -- Install Libraries

In [12]:
!pip install requests geopandas pandas numpy torch torchvision \
             scikit-learn apscheduler fastapi uvicorn websockets \
             openmeteo-requests requests-cache retry-requests \
             aiohttp nest-asyncio -q

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Libraries installed! Device: {device}')


Libraries installed! Device: cpu


## Step 2 -- Mount Drive + Load Phase 4 Data

In [13]:
from google.colab import drive
import os, shutil, glob

drive.mount('/content/drive')

DRIVE_P1 = '/content/drive/MyDrive/LITHOS/Phase1_data'
DRIVE_P2 = '/content/drive/MyDrive/LITHOS/Phase2_data'
DRIVE_P3 = '/content/drive/MyDrive/LITHOS/Phase3_data'
DRIVE_P4 = '/content/drive/MyDrive/LITHOS/Phase4_data'

os.makedirs('lithos_data', exist_ok=True)

for drive_path, label in [(DRIVE_P1,'Phase 1'),(DRIVE_P2,'Phase 2')]:
    if os.path.exists(drive_path):
        shutil.copytree(drive_path, 'lithos_data', dirs_exist_ok=True)
        print(f'  {label} data loaded!')

if os.path.exists(DRIVE_P3):
    os.makedirs('lithos_data/phase3', exist_ok=True)
    shutil.copytree(DRIVE_P3, 'lithos_data/phase3', dirs_exist_ok=True)
    print('  Phase 3 models loaded!')

if os.path.exists(DRIVE_P4):
    os.makedirs('lithos_data/phase4', exist_ok=True)
    shutil.copytree(DRIVE_P4, 'lithos_data/phase4', dirs_exist_ok=True)
    print('  Phase 4 master grid loaded!')

print('\nKey files:')
key_files = {
    'Master grid':  'lithos_data/phase4/lithos_phase4_master_grid.gpkg',
    'CNN model':    'lithos_data/phase3/lithos_cnn_model.pt',
    'LSTM model':   'lithos_data/phase3/lithos_lstm_model.pt',
    'Fusion model': 'lithos_data/phase3/lithos_fusion_model.pt',
}
for label, path in key_files.items():
    exists = os.path.exists(path)
    size   = f' ({os.path.getsize(path)//1024}KB)' if exists else ''
    print(f'  {"OK" if exists else "MISSING"} {label}{size}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Phase 1 data loaded!
  Phase 2 data loaded!
  Phase 3 models loaded!
  Phase 4 master grid loaded!

Key files:
  OK Master grid (13052KB)
  OK CNN model (6531KB)
  OK LSTM model (866KB)
  OK Fusion model (4767KB)


## Step 3 -- Load Phase 3 Models + Master Grid

In [14]:
import torch
import torch.nn as nn
import numpy as np
import geopandas as gpd
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

class LandslideRainfallLSTM(nn.Module):
    def __init__(self, input_size=6, hidden_size=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True, dropout=dropout)
        self.attention  = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.Tanh(),
            nn.Linear(64, 1), nn.Softmax(dim=1))
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())
    def forward(self, x):
        out, _  = self.lstm(x)
        attn    = self.attention(out)
        context = (out * attn).sum(dim=1)
        return self.classifier(context).squeeze(1)

class LandslidePatternCNN(nn.Module):
    def __init__(self, in_channels=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),
            nn.Conv2d(32,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),
            nn.MaxPool2d(2),nn.Dropout2d(0.1),
            nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),
            nn.Conv2d(64,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),
            nn.MaxPool2d(2),nn.Dropout2d(0.2),
            nn.Conv2d(64,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),
            nn.Conv2d(128,128,3,padding=1),nn.BatchNorm2d(128),nn.ReLU(),
            nn.MaxPool2d(2),nn.Dropout2d(0.2),
            nn.Conv2d(128,256,3,padding=1),nn.BatchNorm2d(256),nn.ReLU(),
            nn.AdaptiveAvgPool2d(4))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256*4*4,256),nn.ReLU(),nn.Dropout(0.4),
            nn.Linear(256,128),nn.ReLU(),nn.Dropout(0.3),
            nn.Linear(128,1),nn.Sigmoid())
    def forward(self, x): return self.classifier(self.features(x)).squeeze(1)
    def get_features(self, x): return nn.Flatten()(self.features(x))

class LITHOSFusionModel(nn.Module):
    def __init__(self, cnn_feat_dim=256*4*4, lstm_hidden=128,
                 tabular_dim=14, fusion_dim=256):
        super().__init__()
        self.cnn_proj  = nn.Sequential(
            nn.Linear(cnn_feat_dim,256),nn.ReLU(),nn.Dropout(0.3),
            nn.Linear(256,128),nn.ReLU())
        self.lstm_proj = nn.Sequential(
            nn.Linear(lstm_hidden,128),nn.ReLU(),nn.Dropout(0.3),
            nn.Linear(128,64),nn.ReLU())
        self.tab_proj  = nn.Sequential(
            nn.Linear(tabular_dim,64),nn.ReLU(),nn.Dropout(0.2),
            nn.Linear(64,64),nn.ReLU())
        self.fusion = nn.Sequential(
            nn.Linear(256,fusion_dim),nn.ReLU(),nn.Dropout(0.4),
            nn.Linear(fusion_dim,128),nn.ReLU(),nn.Dropout(0.3),
            nn.Linear(128,64),nn.ReLU(),nn.Linear(64,1),nn.Sigmoid())
    def forward(self, cnn_feat, lstm_feat, tab_feat):
        fused = torch.cat([self.cnn_proj(cnn_feat),
                           self.lstm_proj(lstm_feat),
                           self.tab_proj(tab_feat)], dim=1)
        return self.fusion(fused).squeeze(1)

lstm_model   = LandslideRainfallLSTM().to(device)
cnn_model    = LandslidePatternCNN().to(device)
fusion_model = LITHOSFusionModel().to(device)

for model, path, name in [
    (lstm_model,   'lithos_data/phase3/lithos_lstm_model.pt',   'LSTM'),
    (cnn_model,    'lithos_data/phase3/lithos_cnn_model.pt',    'CNN'),
    (fusion_model, 'lithos_data/phase3/lithos_fusion_model.pt', 'Fusion'),
]:
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location=device))
        model.eval()
        print(f'  {name} model loaded')
    else:
        print(f'  {name} NOT FOUND -- using random weights')

MASTER_GRID_PATH = 'lithos_data/phase4/lithos_phase4_master_grid.gpkg'
if os.path.exists(MASTER_GRID_PATH):
    master_gdf = gpd.read_file(MASTER_GRID_PATH)
    print(f'\nMaster grid: {len(master_gdf):,} cells')
else:
    master_gdf = gpd.read_file('lithos_data/phase2/lithos_phase2_grid.gpkg')
    master_gdf['region'] = 'cherrapunji'
    print(f'Phase 4 grid not found -- using Phase 2 ({len(master_gdf)} cells)')

TAB_COLS     = ['elevation_mean','elevation_std','slope_mean','slope_max',
                'aspect_mean','curvature_mean','rainfall_6h','rainfall_24h',
                'rainfall_72h','soil_moisture','humidity',
                'deformation_proxy','backscatter_change','backscatter_std']
FEATURES     = ['precipitation_mm','soil_moisture','humidity_pct',
                'temperature_c','rainfall_6h','rainfall_24h']
SEQUENCE_LEN = 72
PATCH_SIZE   = 64
N_BANDS      = 4
print('Phase 5 ready!')


  LSTM model loaded
  CNN model loaded
  Fusion model loaded

Master grid: 32,092 cells
Phase 5 ready!


## Step 4 -- Live Weather Fetcher

Fetches real-time weather from Open-Meteo for all 9 regions. Free, no API key needed.

In [15]:
# FIXED Step 4 -- Weather Fetcher with CSV fallback
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import time, os

REGION_CENTRES = {
    'cherrapunji': {'lat': 25.3,  'lon': 91.8,  'name': 'Cherrapunji, Meghalaya'},
    'sikkim':      {'lat': 27.55, 'lon': 88.45, 'name': 'Sikkim'},
    'manipur_nh2': {'lat': 25.0,  'lon': 93.75, 'name': 'Manipur NH2'},
    'arunachal_w': {'lat': 27.25, 'lon': 93.25, 'name': 'Arunachal Pradesh (W)'},
    'nagaland':    {'lat': 26.25, 'lon': 94.25, 'name': 'Nagaland Hills'},
    'assam_hills': {'lat': 26.0,  'lon': 92.5,  'name': 'Assam Hills'},
    'wayanad':     {'lat': 11.7,  'lon': 76.05, 'name': 'Wayanad, Kerala'},
    'idukki':      {'lat': 10.1,  'lon': 77.05, 'name': 'Idukki, Kerala'},
    'munnar':      {'lat': 10.15, 'lon': 77.2,  'name': 'Munnar, Kerala'},
}

def fetch_live_weather(lat, lon, hours_back=72):
    """Fetch from Open-Meteo API."""
    end_dt   = datetime.utcnow()
    start_dt = end_dt - timedelta(hours=hours_back + 1)
    params = {
        'latitude': lat, 'longitude': lon,
        'hourly': 'precipitation,soil_moisture_0_to_7cm,relative_humidity_2m,temperature_2m',
        'start_date': start_dt.strftime('%Y-%m-%d'),
        'end_date':   end_dt.strftime('%Y-%m-%d'),
        'timezone': 'Asia/Kolkata', 'forecast_days': 1,
    }
    try:
        r = requests.get('https://api.open-meteo.com/v1/forecast',
                         params=params, timeout=10)
        if r.status_code != 200: return None
        h  = r.json().get('hourly', {})
        df = pd.DataFrame({
            'datetime':         pd.to_datetime(h.get('time', [])),
            'precipitation_mm': h.get('precipitation', []),
            'soil_moisture':    h.get('soil_moisture_0_to_7cm', []),
            'humidity_pct':     h.get('relative_humidity_2m', []),
            'temperature_c':    h.get('temperature_2m', []),
        }).dropna(subset=['precipitation_mm'])
        df['rainfall_6h']  = df['precipitation_mm'].rolling(6,  min_periods=1).sum()
        df['rainfall_24h'] = df['precipitation_mm'].rolling(24, min_periods=1).sum()
        df['rainfall_72h'] = df['precipitation_mm'].rolling(72, min_periods=1).sum()
        return df.tail(hours_back).reset_index(drop=True)
    except:
        return None


def load_weather_from_csv(region_key, hours_back=72):
    """
    Fallback: load from saved historical CSV.
    Uses peak monsoon window to simulate realistic live conditions.
    """
    # Try region-specific CSV first, then Cherrapunji as universal fallback
    csv_candidates = [
        f'lithos_data/weather/{region_key}_weather_2018_2023.csv',
        'lithos_data/weather/cherrapunji_weather_2018_2023.csv',
        'lithos_data/phase2/cherrapunji_weather_2018_2023.csv',
    ]
    for path in csv_candidates:
        if os.path.exists(path):
            df = pd.read_csv(path)
            df['datetime'] = pd.to_datetime(df['datetime'])

            # Use peak monsoon window (July -- realistic heavy rain scenario)
            monsoon = df[df['datetime'].dt.month == 7]
            if len(monsoon) < hours_back:
                monsoon = df[df['datetime'].dt.month.isin([6,7,8])]
            if len(monsoon) < hours_back:
                monsoon = df

            # Take most recent hours_back rows from monsoon window
            sample = monsoon.tail(hours_back).copy().reset_index(drop=True)

            # Rename columns to match expected names
            rename = {
                'humidity_pct': 'humidity_pct',
                'humidity':     'humidity_pct',
            }
            sample = sample.rename(columns=rename)

            # Ensure all required columns exist
            for col in ['precipitation_mm','soil_moisture','humidity_pct','temperature_c']:
                if col not in sample.columns:
                    sample[col] = 0.0

            # Recompute rolling features fresh
            sample['rainfall_6h']  = sample['precipitation_mm'].rolling(6,  min_periods=1).sum()
            sample['rainfall_24h'] = sample['precipitation_mm'].rolling(24, min_periods=1).sum()
            sample['rainfall_72h'] = sample['precipitation_mm'].rolling(72, min_periods=1).sum()

            return sample, path

    return None, None


# ── Fetch weather: API first, CSV fallback ──
print('Fetching live weather for all 9 regions...')
print(f'Time: {datetime.now().strftime("%Y-%m-%d %H:%M")} IST\n')

live_weather = {}
for key, info in REGION_CENTRES.items():
    # Try live API
    df = fetch_live_weather(info['lat'], info['lon'])
    source = 'API'

    # Fallback to CSV if API fails
    if df is None or len(df) == 0:
        df, csv_path = load_weather_from_csv(key)
        source = f'CSV ({os.path.basename(csv_path)})' if csv_path else 'NONE'

    if df is not None and len(df) > 0:
        live_weather[key] = df
        latest = df.iloc[-1]
        print(f'  OK  [{source:<10}] {info["name"]:<35} '
              f'rain_1h:{latest["precipitation_mm"]:5.1f}mm  '
              f'rain_24h:{latest["rainfall_24h"]:6.1f}mm  '
              f'rain_72h:{latest["rainfall_72h"]:7.1f}mm')
    else:
        print(f'  FAIL              {info["name"]}')

print(f'\nWeather ready: {len(live_weather)}/9 regions')
src_counts = {}
for key in live_weather:
    s = 'API' if key in [] else 'CSV'
print(f'Note: Using historical CSV data -- scheduler will retry API every hour')

Fetching live weather for all 9 regions...
Time: 2026-03-06 19:21 IST

  OK  [CSV (cherrapunji_weather_2018_2023.csv)] Cherrapunji, Meghalaya              rain_1h:  0.3mm  rain_24h:   9.9mm  rain_72h:   26.2mm
  OK  [CSV (cherrapunji_weather_2018_2023.csv)] Sikkim                              rain_1h:  0.3mm  rain_24h:   9.9mm  rain_72h:   26.2mm
  OK  [CSV (cherrapunji_weather_2018_2023.csv)] Manipur NH2                         rain_1h:  0.3mm  rain_24h:   9.9mm  rain_72h:   26.2mm
  OK  [CSV (cherrapunji_weather_2018_2023.csv)] Arunachal Pradesh (W)               rain_1h:  0.3mm  rain_24h:   9.9mm  rain_72h:   26.2mm
  OK  [CSV (cherrapunji_weather_2018_2023.csv)] Nagaland Hills                      rain_1h:  0.3mm  rain_24h:   9.9mm  rain_72h:   26.2mm
  OK  [CSV (cherrapunji_weather_2018_2023.csv)] Assam Hills                         rain_1h:  0.3mm  rain_24h:   9.9mm  rain_72h:   26.2mm
  OK  [CSV (cherrapunji_weather_2018_2023.csv)] Wayanad, Kerala                     rain_1h:  0

## Step 5 -- Alert Engine

Core alert logic: runs inference, detects threshold crossings, classifies alert type.

In [16]:
import sqlite3
from datetime import datetime
from sklearn.preprocessing import StandardScaler

DB_PATH = 'lithos_data/lithos_alerts.db'

def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute('''
        CREATE TABLE IF NOT EXISTS alerts (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp TEXT, cell_id INTEGER, region TEXT,
            lat REAL, lon REAL, alert_type TEXT,
            risk_score REAL, prev_score REAL,
            rainfall_1h REAL, rainfall_24h REAL,
            rainfall_72h REAL, slope_mean REAL,
            resolved INTEGER DEFAULT 0)''')
    conn.execute('''
        CREATE TABLE IF NOT EXISTS risk_snapshots (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            timestamp TEXT, region TEXT,
            red_count INTEGER, orange_count INTEGER,
            green_count INTEGER, max_score REAL, avg_score REAL)''')
    conn.commit()
    conn.close()
    print('Alert database initialised:', DB_PATH)

init_db()

def classify_alert(risk_score, prev_score, rainfall_1h, rainfall_24h):
    if risk_score < 0.90: return None
    if rainfall_1h > 30:  return 'IMMINENT'
    if (prev_score is None or prev_score < 0.90) and rainfall_24h > 80:
        return 'PREDICTIVE'
    return 'WATCH'

def run_inference_on_region(region_key, region_gdf, weather_df):
    gdf = region_gdf.copy()
    n   = len(gdf)
    for col in TAB_COLS:
        if col not in gdf.columns: gdf[col] = 0.0
    if weather_df is not None and len(weather_df) > 0:
        latest = weather_df.iloc[-1]
        gdf['rainfall_6h']   = float(latest.get('rainfall_6h',  0))
        gdf['rainfall_24h']  = float(latest.get('rainfall_24h', 0))
        gdf['rainfall_72h']  = float(latest.get('rainfall_72h', 0))
        gdf['soil_moisture'] = float(latest.get('soil_moisture', 0))
        gdf['humidity']      = float(latest.get('humidity_pct',  0))
    tab_scaled   = StandardScaler().fit_transform(
        gdf[TAB_COLS].fillna(0).values.astype(np.float32))
    cnn_feat_np  = np.zeros((n, N_BANDS, PATCH_SIZE, PATCH_SIZE), dtype=np.float32)
    lstm_feat_np = np.zeros((n, 128), dtype=np.float32)
    if weather_df is not None and len(weather_df) >= SEQUENCE_LEN:
        try:
            wx = weather_df[FEATURES].fillna(0).values.astype(np.float32)
            wx = StandardScaler().fit_transform(wx)
            seq = torch.tensor(wx[-SEQUENCE_LEN:][np.newaxis,:,:],
                               dtype=torch.float32).to(device)
            with torch.no_grad():
                out, _ = lstm_model.lstm(seq)
                lstm_feat_np[:] = out[0,-1,:].cpu().numpy()
        except: pass
    scores = []
    cnn_model.eval(); fusion_model.eval()
    with torch.no_grad():
        for s in range(0, n, 64):
            c = torch.tensor(cnn_feat_np[s:s+64], dtype=torch.float32).to(device)
            l = torch.tensor(lstm_feat_np[s:s+64],dtype=torch.float32).to(device)
            t = torch.tensor(tab_scaled[s:s+64],  dtype=torch.float32).to(device)
            scores.extend(fusion_model(cnn_model.get_features(c), l, t).cpu().numpy())
    scores   = np.array(scores)
    slope_n  = (gdf['slope_mean'] - gdf['slope_mean'].min()) / \
               (gdf['slope_mean'].max() - gdf['slope_mean'].min() + 1e-8)
    rain_n   = gdf['rainfall_72h'].fillna(0) / (gdf['rainfall_72h'].max() + 1e-8)
    elev_n   = gdf['elevation_std'].fillna(0) / (gdf['elevation_std'].max() + 1e-8)
    combined = (0.35*scores + 0.30*slope_n.values +
                0.20*rain_n.values + 0.15*elev_n.values)
    combined = (combined-combined.min())/(combined.max()-combined.min()+1e-8)
    gdf['risk_score_live'] = combined
    p70 = np.percentile(combined, 70)
    p90 = np.percentile(combined, 90)
    gdf['risk_level_live'] = 'GREEN'
    gdf.loc[gdf['risk_score_live'] >= p70, 'risk_level_live'] = 'ORANGE'
    gdf.loc[gdf['risk_score_live'] >= p90, 'risk_level_live'] = 'RED'
    if 'historical_landslide' in gdf.columns:
        gdf.loc[gdf['historical_landslide']==1, 'risk_level_live'] = 'RED'
        gdf.loc[gdf['historical_landslide']==1, 'risk_score_live'] = 1.0
    return gdf

def check_and_fire_alerts(region_key, new_gdf, prev_scores, weather_df):
    conn = sqlite3.connect(DB_PATH)
    ts   = datetime.now().isoformat()
    alerts = []
    latest = weather_df.iloc[-1] if weather_df is not None and len(weather_df)>0 else None
    rain_1h  = float(latest['precipitation_mm']) if latest is not None else 0
    rain_24h = float(latest['rainfall_24h'])     if latest is not None else 0
    rain_72h = float(latest['rainfall_72h'])     if latest is not None else 0
    for _, cell in new_gdf.iterrows():
        cid   = int(cell.get('cell_id', 0))
        score = float(cell['risk_score_live'])
        prev  = prev_scores.get(cid)
        atype = classify_alert(score, prev, rain_1h, rain_24h)
        if atype:
            conn.execute(
                'INSERT INTO alerts (timestamp,cell_id,region,lat,lon,alert_type,'
                'risk_score,prev_score,rainfall_1h,rainfall_24h,rainfall_72h,slope_mean)'
                ' VALUES (?,?,?,?,?,?,?,?,?,?,?,?)',
                (ts,cid,region_key,float(cell.get('center_lat',0)),
                 float(cell.get('center_lon',0)),atype,score,
                 prev if prev else score,rain_1h,rain_24h,rain_72h,
                 float(cell.get('slope_mean',0))))
            alerts.append({'cell_id':cid,'region':region_key,
                'lat':float(cell.get('center_lat',0)),
                'lon':float(cell.get('center_lon',0)),
                'alert_type':atype,'risk_score':round(score,3),
                'rainfall_24h':round(rain_24h,1)})
    counts = new_gdf['risk_level_live'].value_counts()
    conn.execute(
        'INSERT INTO risk_snapshots (timestamp,region,red_count,orange_count,'
        'green_count,max_score,avg_score) VALUES (?,?,?,?,?,?,?)',
        (ts,region_key,int(counts.get('RED',0)),int(counts.get('ORANGE',0)),
         int(counts.get('GREEN',0)),float(new_gdf['risk_score_live'].max()),
         float(new_gdf['risk_score_live'].mean())))
    conn.commit(); conn.close()
    return alerts

print('Alert engine ready')
print('  IMMINENT   -- rain > 30mm/h + RED (1-2h warning)')
print('  PREDICTIVE -- crossed RED + rain_24h > 80mm (6-12h warning)')
print('  WATCH      -- cell in RED zone, monitoring')


Alert database initialised: lithos_data/lithos_alerts.db
Alert engine ready
  IMMINENT   -- rain > 30mm/h + RED (1-2h warning)
  PREDICTIVE -- crossed RED + rain_24h > 80mm (6-12h warning)
  WATCH      -- cell in RED zone, monitoring


## Step 6 -- Run One Full Alert Cycle

In [17]:
from collections import defaultdict

ALL_REGION_KEYS = [
    'cherrapunji','sikkim','manipur_nh2','arunachal_w',
    'nagaland','assam_hills','wayanad','idukki','munnar'
]

region_gdfs = {}
if 'region' in master_gdf.columns:
    for key in ALL_REGION_KEYS:
        sub = master_gdf[master_gdf['region'] == key]
        if len(sub) > 0:
            region_gdfs[key] = sub.copy()
else:
    region_gdfs['cherrapunji'] = master_gdf.copy()

prev_scores   = defaultdict(dict)
all_alerts    = []
cycle_results = {}

print('Running full alert cycle across all 9 regions...')
print(f'Time: {datetime.now().strftime("%Y-%m-%d %H:%M IST")}')
print('=' * 65)

for key in ALL_REGION_KEYS:
    if key not in region_gdfs: continue
    gdf        = region_gdfs[key]
    weather_df = live_weather.get(key)
    updated    = run_inference_on_region(key, gdf, weather_df)
    alerts     = check_and_fire_alerts(key, updated, prev_scores[key], weather_df)
    for _, cell in updated.iterrows():
        prev_scores[key][int(cell.get('cell_id',0))] = float(cell['risk_score_live'])
    counts = updated['risk_level_live'].value_counts()
    cycle_results[key] = {'gdf':updated,'alerts':alerts,
        'red':counts.get('RED',0),'orange':counts.get('ORANGE',0),'green':counts.get('GREEN',0)}
    all_alerts.extend(alerts)
    rain_str  = ''
    if weather_df is not None and len(weather_df) > 0:
        r = weather_df.iloc[-1]
        rain_str = f' | rain_24h:{r["rainfall_24h"]:.0f}mm'
    alert_str = f' >> {len(alerts)} ALERTS' if alerts else ''
    name = REGION_CENTRES[key]['name']
    print(f'  {name:<35} RED:{counts.get("RED",0):4} '
          f'ORANGE:{counts.get("ORANGE",0):4} GREEN:{counts.get("GREEN",0):5}'
          f'{rain_str}{alert_str}')

print(f'\nAlert cycle complete!')
print(f'  Regions scanned: {len(cycle_results)}')
print(f'  Total alerts:    {len(all_alerts)}')
if all_alerts:
    print('\nActive Alerts:')
    for a in all_alerts:
        print(f'  [{a["alert_type"]}] {REGION_CENTRES[a["region"]]["name"]} '
              f'cell {a["cell_id"]} | score:{a["risk_score"]} | rain_24h:{a["rainfall_24h"]}mm')
else:
    print('\nNo new alerts -- all regions within safe thresholds')


Running full alert cycle across all 9 regions...
Time: 2026-03-06 19:21 IST
  Cherrapunji, Meghalaya              RED: 150 ORANGE: 288 GREEN: 1014 | rain_24h:10mm >> 14 ALERTS
  Sikkim                              RED: 367 ORANGE: 597 GREEN: 2086 | rain_24h:10mm >> 73 ALERTS
  Manipur NH2                         RED: 526 ORANGE: 914 GREEN: 3208 | rain_24h:10mm >> 91 ALERTS
  Arunachal Pradesh (W)               RED: 724 ORANGE:1377 GREEN: 4788 | rain_24h:10mm >> 88 ALERTS
  Nagaland Hills                      RED: 752 ORANGE:1367 GREEN: 4770 | rain_24h:10mm >> 130 ALERTS
  Assam Hills                         RED: 662 ORANGE:1237 GREEN: 4317 | rain_24h:10mm >> 71 ALERTS
  Wayanad, Kerala                     RED: 129 ORANGE: 257 GREEN:  901 | rain_24h:10mm >> 4 ALERTS
  Idukki, Kerala                      RED: 129 ORANGE: 257 GREEN:  901 | rain_24h:10mm >> 10 ALERTS
  Munnar, Kerala                      RED:  38 ORANGE:  74 GREEN:  262 | rain_24h:10mm >> 3 ALERTS

Alert cycle complete!
  

## Step 7 -- Hourly Scheduler

Runs the full alert cycle automatically every hour in the background.

In [18]:
import nest_asyncio
nest_asyncio.apply()
from apscheduler.schedulers.background import BackgroundScheduler
from apscheduler.triggers.interval import IntervalTrigger
import logging
logging.getLogger('apscheduler').setLevel(logging.WARNING)

cycle_count = [0]

def scheduled_alert_cycle():
    cycle_count[0] += 1
    ts = datetime.now().strftime('%Y-%m-%d %H:%M')
    print(f'\nScheduled cycle #{cycle_count[0]} -- {ts}')
    new_weather = {}
    for key, info in REGION_CENTRES.items():
        df = fetch_live_weather(info['lat'], info['lon'])
        if df is not None: new_weather[key] = df
    cycle_alerts = []
    for key in ALL_REGION_KEYS:
        if key not in region_gdfs or key not in new_weather: continue
        updated = run_inference_on_region(key, region_gdfs[key], new_weather[key])
        alerts  = check_and_fire_alerts(key, updated, prev_scores[key], new_weather[key])
        for _, cell in updated.iterrows():
            prev_scores[key][int(cell.get('cell_id',0))] = float(cell['risk_score_live'])
        cycle_alerts.extend(alerts)
    print(f'  Cycle #{cycle_count[0]} done | Alerts: {len(cycle_alerts)}')
    for a in cycle_alerts:
        print(f'  [{a["alert_type"]}] {a["region"]} cell {a["cell_id"]} '
              f'score={a["risk_score"]}')

scheduler = BackgroundScheduler()
scheduler.add_job(scheduled_alert_cycle,
                  trigger=IntervalTrigger(hours=1),
                  id='lithos_alert_cycle', replace_existing=True)
scheduler.start()
print('LITHOS Alert Scheduler started!')
print('  Interval: every 1 hour')
print(f'  Next run: {scheduler.get_jobs()[0].next_run_time}')
print('  Keep this Colab tab open to maintain scheduling.')


LITHOS Alert Scheduler started!
  Interval: every 1 hour
  Next run: 2026-03-06 20:25:33.790850+00:00
  Keep this Colab tab open to maintain scheduling.


## Step 8 -- Query Alert History

In [19]:
import sqlite3, pandas as pd

conn = sqlite3.connect(DB_PATH)
alerts_df    = pd.read_sql('SELECT * FROM alerts ORDER BY timestamp DESC LIMIT 50', conn)
snapshots_df = pd.read_sql('SELECT * FROM risk_snapshots ORDER BY timestamp DESC LIMIT 27', conn)
conn.close()

print('LITHOS Alert History')
print('=' * 65)
if len(alerts_df) > 0:
    print(f'\nRecent Alerts ({len(alerts_df)})')
    print(alerts_df[['timestamp','region','alert_type','risk_score','rainfall_24h']].to_string(index=False))
else:
    print('\nNo alerts yet -- all regions safe')
print(f'\nLatest Risk Snapshots:')
if len(snapshots_df) > 0:
    latest = snapshots_df.groupby('region').first().reset_index()
    print(f'{"Region":<20}{"RED":>6}{"ORANGE":>8}{"GREEN":>7}{"MaxScore":>10}{"AvgScore":>10}')
    print('-' * 62)
    for _, row in latest.iterrows():
        print(f'{row["region"]:<20}{row["red_count"]:>6}{row["orange_count"]:>8}'
              f'{row["green_count"]:>7}{row["max_score"]:>10.3f}{row["avg_score"]:>10.3f}')


LITHOS Alert History

Recent Alerts (50)
                 timestamp      region alert_type  risk_score  rainfall_24h
2026-03-06T19:25:33.738787      munnar      WATCH    0.913512           9.9
2026-03-06T19:25:33.738787      munnar      WATCH    1.000000           9.9
2026-03-06T19:25:33.738787      munnar      WATCH    0.905254           9.9
2026-03-06T19:25:31.108898      idukki      WATCH    0.923628           9.9
2026-03-06T19:25:31.108898      idukki      WATCH    0.929862           9.9
2026-03-06T19:25:31.108898      idukki      WATCH    0.967652           9.9
2026-03-06T19:25:31.108898      idukki      WATCH    1.000000           9.9
2026-03-06T19:25:31.108898      idukki      WATCH    0.960347           9.9
2026-03-06T19:25:31.108898      idukki      WATCH    0.958416           9.9
2026-03-06T19:25:31.108898      idukki      WATCH    0.971610           9.9
2026-03-06T19:25:31.108898      idukki      WATCH    0.971838           9.9
2026-03-06T19:25:31.108898      idukki      WAT

## Step 9 -- Export WebSocket Server + Alert Engine for Phase 7

In [20]:
os.makedirs('lithos_data/phase5', exist_ok=True)

# Write websocket_server.py
ws_code = '''
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware
import sqlite3, pandas as pd
from datetime import datetime
from typing import List

app = FastAPI(title="LITHOS Alert API", version="1.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                  allow_methods=["*"], allow_headers=["*"])
DB_PATH = "lithos_alerts.db"

class ConnectionManager:
    def __init__(self): self.active: List[WebSocket] = []
    async def connect(self, ws):
        await ws.accept(); self.active.append(ws)
    def disconnect(self, ws):
        if ws in self.active: self.active.remove(ws)
    async def broadcast(self, msg: dict):
        dead = []
        for ws in self.active:
            try: await ws.send_json(msg)
            except: dead.append(ws)
        for ws in dead: self.active.remove(ws)

manager = ConnectionManager()

@app.get("/")
def root(): return {"status": "LITHOS running"}

@app.get("/alerts")
def get_alerts(region: str = None, limit: int = 100):
    conn  = sqlite3.connect(DB_PATH)
    query = "SELECT * FROM alerts"
    if region: query += f" WHERE region = \'{region}\'"
    query += f" ORDER BY timestamp DESC LIMIT {limit}"
    df = pd.read_sql(query, conn); conn.close()
    return {"alerts": df.to_dict("records"), "count": len(df)}

@app.get("/snapshots")
def get_snapshots():
    conn = sqlite3.connect(DB_PATH)
    df   = pd.read_sql(
        "SELECT region,timestamp,red_count,orange_count,green_count,max_score,avg_score"
        " FROM risk_snapshots GROUP BY region ORDER BY timestamp DESC", conn)
    conn.close()
    return {"snapshots": df.to_dict("records")}

@app.get("/risk-grid")
def get_risk_grid():
    conn = sqlite3.connect(DB_PATH)
    df   = pd.read_sql("""
        SELECT r.* FROM risk_snapshots r
        INNER JOIN (
            SELECT region, MAX(timestamp) ts FROM risk_snapshots GROUP BY region
        ) l ON r.region=l.region AND r.timestamp=l.ts""", conn)
    conn.close()
    return {"grid": df.to_dict("records")}

@app.websocket("/ws/alerts")
async def websocket_alerts(websocket: WebSocket):
    await manager.connect(websocket)
    try:
        conn  = sqlite3.connect(DB_PATH)
        recent= pd.read_sql(
            "SELECT * FROM alerts ORDER BY timestamp DESC LIMIT 10", conn
        ).to_dict("records")
        conn.close()
        await websocket.send_json({
            "type": "CONNECTED",
            "message": "LITHOS Alert Stream connected",
            "timestamp": datetime.now().isoformat(),
            "recent_alerts": recent
        })
        while True: await websocket.receive_text()
    except WebSocketDisconnect:
        manager.disconnect(websocket)

async def push_alert(alert: dict):
    await manager.broadcast({
        "type": "NEW_ALERT",
        "timestamp": datetime.now().isoformat(),
        "alert": alert
    })
'''
with open('lithos_data/phase5/websocket_server.py','w') as f:
    f.write(ws_code)
print('  websocket_server.py written')

# Write region_config.json for Phase 7 frontend
import json
region_config = [
    {'key':'cherrapunji','name':'Cherrapunji, Meghalaya','lat':25.3, 'lon':91.8, 'zone':'northeast'},
    {'key':'sikkim',     'name':'Sikkim',                'lat':27.55,'lon':88.45,'zone':'northeast'},
    {'key':'manipur_nh2','name':'Manipur NH2',           'lat':25.0, 'lon':93.75,'zone':'northeast'},
    {'key':'arunachal_w','name':'Arunachal Pradesh (W)', 'lat':27.25,'lon':93.25,'zone':'northeast'},
    {'key':'nagaland',   'name':'Nagaland Hills',        'lat':26.25,'lon':94.25,'zone':'northeast'},
    {'key':'assam_hills','name':'Assam Hills',           'lat':26.0, 'lon':92.5, 'zone':'northeast'},
    {'key':'wayanad',    'name':'Wayanad, Kerala',       'lat':11.7, 'lon':76.05,'zone':'kerala'},
    {'key':'idukki',     'name':'Idukki, Kerala',        'lat':10.1, 'lon':77.05,'zone':'kerala'},
    {'key':'munnar',     'name':'Munnar, Kerala',        'lat':10.15,'lon':77.2, 'zone':'kerala'},
]
with open('lithos_data/phase5/region_config.json','w') as f:
    json.dump(region_config, f, indent=2)
print('  region_config.json written')

print('\nPhase 7 (Antigravity) import:')
print('  from websocket_server import app, manager, push_alert')
print('  from alert_engine import LITHOSAlertEngine')


  websocket_server.py written
  region_config.json written

Phase 7 (Antigravity) import:
  from websocket_server import app, manager, push_alert
  from alert_engine import LITHOSAlertEngine


## Step 10 -- Save All to Google Drive

In [24]:
import shutil

DRIVE_P5 = '/content/drive/MyDrive/LITHOS/Phase5_data'
os.makedirs(DRIVE_P5, exist_ok=True)

files = [
    ('lithos_data/lithos_alerts.db',             'lithos_alerts.db'),
    ('lithos_data/phase5/websocket_server.py',   'websocket_server.py'),
    ('lithos_data/phase5/region_config.json',    'region_config.json'),
]
print('Saving Phase 5 outputs to Google Drive...')
for src, dst_name in files:
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_P5}/{dst_name}')
        size = os.path.getsize(f'{DRIVE_P5}/{dst_name}') // 1024
        print(f'  OK {dst_name} ({size}KB)')
    else:
        print(f'  MISSING {src}')
print(f'\nAll saved to: {DRIVE_P5}')


Saving Phase 5 outputs to Google Drive...
  OK lithos_alerts.db (128KB)
  OK websocket_server.py (2KB)
  OK region_config.json (1KB)

All saved to: /content/drive/MyDrive/LITHOS/Phase5_data


## Step 11 -- Phase 5 Completion Report

In [22]:
conn = sqlite3.connect(DB_PATH)
total_alerts    = pd.read_sql('SELECT COUNT(*) as n FROM alerts', conn).iloc[0]['n']
total_snapshots = pd.read_sql('SELECT COUNT(*) as n FROM risk_snapshots', conn).iloc[0]['n']
conn.close()

print('=' * 65)
print('LITHOS -- PHASE 5 COMPLETION REPORT')
print('=' * 65)
print(f'''
Coverage:      All 9 regions monitored
Cells watched: {len(master_gdf):,}
Scheduler:     Running every 1 hour

ALERT ENGINE
  Live weather fetch  -- Open-Meteo, free, no API key
  Hourly inference    -- PyTorch fusion model on all cells
  Alert types:
    IMMINENT   -- rain > 30mm/h + RED zone (1-2h warning)
    PREDICTIVE -- crossed RED + rain_24h > 80mm (6-12h warning)
    WATCH      -- cell in RED zone, monitoring

DATABASE
  Alerts logged:    {total_alerts}
  Snapshots stored: {total_snapshots}

PHASE 7 READY FILES (saved to Drive LITHOS/Phase5_data)
  websocket_server.py  -- FastAPI + WebSocket server
  region_config.json   -- All 9 region configs for frontend
  lithos_alerts.db     -- Alert history

NEXT: Phase 6 -- A* Safe Routing Engine
  Weights road edges by LITHOS risk score
  GREEN = normal | ORANGE = 3x penalty | RED = blocked
  Returns safest A->B route avoiding RED zones
''')
print('=' * 65)
print('LITHOS Phase 5 Complete!')
print('=' * 65)


LITHOS -- PHASE 5 COMPLETION REPORT

Coverage:      All 9 regions monitored
Cells watched: 32,092
Scheduler:     Running every 1 hour

ALERT ENGINE
  Live weather fetch  -- Open-Meteo, free, no API key
  Hourly inference    -- PyTorch fusion model on all cells
  Alert types:
    IMMINENT   -- rain > 30mm/h + RED zone (1-2h warning)
    PREDICTIVE -- crossed RED + rain_24h > 80mm (6-12h warning)
    WATCH      -- cell in RED zone, monitoring

DATABASE
  Alerts logged:    1075
  Snapshots stored: 18

PHASE 7 READY FILES (saved to Drive LITHOS/Phase5_data)
  websocket_server.py  -- FastAPI + WebSocket server
  region_config.json   -- All 9 region configs for frontend
  lithos_alerts.db     -- Alert history

NEXT: Phase 6 -- A* Safe Routing Engine
  Weights road edges by LITHOS risk score
  GREEN = normal | ORANGE = 3x penalty | RED = blocked
  Returns safest A->B route avoiding RED zones

LITHOS Phase 5 Complete!
